# Flipkart Customer Satisfaction EDA

**Project type:** Exploratory Data Analysis  
**Dataset:** `Customer_support_data.csv`  
**Business theme:** Understand the drivers of customer satisfaction (CSAT) across support channels, issue types, product categories, handling time, agents, shifts, tenure groups, and customer cities.


## Project Summary

This notebook performs an end-to-end exploratory data analysis for Flipkart customer support satisfaction data. The analysis starts with a data audit covering dataset shape, column names, data types, descriptive statistics, missing values, duplicates, and date parsing. It then studies CSAT Score patterns across customer support channels, issue categories, product categories, connected handling time, response time, agent performance, agent shifts, tenure buckets, and customer city trends.

The objective is to move beyond a simple score distribution and identify operational levers that can improve customer experience. The notebook includes more than ten business-focused visualizations using Matplotlib and Seaborn. Each visualization is followed by markdown business insights so the evaluation narrative is clear and submission-ready.


## Problem Statement

Flipkart receives customer support interactions through multiple channels and across many issue types. The business needs to understand what factors are associated with higher or lower CSAT scores so it can prioritize service quality improvements, coach agents, and identify segments where dissatisfaction is concentrated.


## Business Objectives

- Measure the overall CSAT distribution and identify the share of low, neutral, and high satisfaction responses.
- Compare CSAT performance across channel, issue category, product category, handling time, agent, tenure, shift, and city dimensions.
- Identify high-performing and low-performing operational segments.
- Highlight data quality limitations that affect interpretation.
- Provide actionable recommendations for customer support leadership.


# 1. Import Libraries


In [ ]:
# Core data libraries
import numpy as np
import pandas as pd

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Display and warnings
import warnings
warnings.filterwarnings("ignore")

# Plot styling
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (11, 6)
plt.rcParams["axes.titlesize"] = 15
plt.rcParams["axes.labelsize"] = 11
plt.rcParams["xtick.labelsize"] = 10
plt.rcParams["ytick.labelsize"] = 10
plt.rcParams["figure.dpi"] = 110

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)


# 2. Load Dataset


In [ ]:
# Load the dataset
file_path = "Customer_support_data.csv"
df = pd.read_csv(file_path)

# Keep an untouched copy for reference
raw_df = df.copy()

df.head()


# 3. Know Your Data


In [ ]:
# Dataset shape
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]:,}")


In [ ]:
# Column names
list(df.columns)


In [ ]:
# Data types and non-null counts
df.info()


In [ ]:
# First five rows
df.head()


In [ ]:
# Last five rows
df.tail()


# 4. Descriptive Statistics


In [ ]:
# Descriptive statistics for numeric columns
df.describe().T


In [ ]:
# Descriptive statistics for object/categorical columns
df.describe(include="object").T


# 5. Missing Values and Duplicate Records


In [ ]:
# Missing value audit
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)
missing_summary


In [ ]:
# Duplicate row audit
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows:,}")


In [ ]:
# Unique values per column
unique_summary = pd.DataFrame({
    "unique_values": df.nunique(dropna=True),
    "non_null_count": df.notna().sum()
}).sort_values("unique_values", ascending=False)
unique_summary


## Data Quality Insight

The target column `CSAT Score` is complete, which is good for satisfaction analysis. However, several explanatory fields are sparse, especially `connected_handling_time`, `Customer_City`, `Product_category`, `Item_price`, and `order_date_time`. Visuals using these fields should therefore be interpreted as segment-level indicators from available records, not full-population conclusions. No full-row duplicates are present.


# 6. Data Preparation for EDA


In [ ]:
# Parse date/time columns.
df["Issue_reported at"] = pd.to_datetime(df["Issue_reported at"], errors="coerce", dayfirst=True)
df["issue_responded"] = pd.to_datetime(df["issue_responded"], errors="coerce", dayfirst=True)
df["Survey_response_Date"] = pd.to_datetime(df["Survey_response_Date"], errors="coerce", dayfirst=True)

# Derive response time from issue reported and responded timestamps.
df["response_time_minutes"] = (
    df["issue_responded"] - df["Issue_reported at"]
).dt.total_seconds() / 60

# Keep only operationally valid non-negative response times for response-time analysis.
df["valid_response_time_minutes"] = df["response_time_minutes"].where(df["response_time_minutes"] >= 0)

# Customer satisfaction groups for business interpretation.
df["CSAT_Group"] = pd.cut(
    df["CSAT Score"],
    bins=[0, 2, 3, 5],
    labels=["Low (1-2)", "Neutral (3)", "High (4-5)"],
    include_lowest=True
)

# Create a daily date field for trend analysis.
df["survey_day"] = df["Survey_response_Date"].dt.date

df[["CSAT Score", "CSAT_Group", "response_time_minutes", "valid_response_time_minutes", "survey_day"]].head()


# 7. Overall CSAT Analysis


## Visualization 1: Distribution of CSAT Score


In [ ]:
plt.close("all")
# Count distribution of CSAT Score
csat_order = sorted(df["CSAT Score"].dropna().unique())

ax = sns.countplot(data=df, x="CSAT Score", order=csat_order, palette="viridis")
ax.set_title("Distribution of CSAT Score")
ax.set_xlabel("CSAT Score")
ax.set_ylabel("Number of Responses")

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()


**Business Insight:** CSAT is strongly skewed toward positive ratings. Score 5 accounts for the largest share of responses, while scores 1 and 4 form the next major groups. This indicates generally healthy satisfaction, but the sizeable count of score 1 responses should be treated as a priority dissatisfaction pool for root-cause analysis.


## Visualization 2: CSAT Sentiment Group Share


In [ ]:
plt.close("all")
# Business-friendly satisfaction grouping
csat_group_share = (
    df["CSAT_Group"]
    .value_counts(normalize=True)
    .reindex(["Low (1-2)", "Neutral (3)", "High (4-5)"])
    .mul(100)
    .reset_index()
)
csat_group_share.columns = ["CSAT_Group", "Percentage"]

ax = sns.barplot(data=csat_group_share, x="CSAT_Group", y="Percentage", palette="Set2")
ax.set_title("Share of Low, Neutral, and High CSAT Responses")
ax.set_xlabel("CSAT Group")
ax.set_ylabel("Percentage of Responses")
ax.set_ylim(0, 100)

for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=3)

plt.tight_layout()
plt.show()


**Business Insight:** High satisfaction responses dominate the dataset, but low CSAT responses remain large enough to represent meaningful customer pain. Tracking the low-CSAT share over time is more actionable than only monitoring the average CSAT score.


# 8. Channel Analysis


## Visualization 3: Support Channel Volume


In [ ]:
plt.close("all")
# Channel-wise contact volume
channel_counts = df["channel_name"].value_counts().reset_index()
channel_counts.columns = ["channel_name", "count"]

ax = sns.barplot(data=channel_counts, x="channel_name", y="count", palette="Set3")
ax.set_title("Support Interaction Volume by Channel")
ax.set_xlabel("Channel")
ax.set_ylabel("Number of Interactions")

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()


**Business Insight:** Inbound support carries the highest interaction load by a wide margin, followed by outcall and email. Any operational improvement in inbound workflows will likely have the largest effect on total customer experience.


## Visualization 4: Channel Name vs Average CSAT Score


In [ ]:
plt.close("all")
# Average CSAT by support channel
channel_csat = (
    df.groupby("channel_name", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
    .sort_values("avg_csat", ascending=False)
)

ax = sns.barplot(data=channel_csat, x="channel_name", y="avg_csat", palette="mako")
ax.set_title("Average CSAT Score by Channel")
ax.set_xlabel("Channel")
ax.set_ylabel("Average CSAT Score")
ax.set_ylim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
channel_csat


**Business Insight:** Outcall and inbound channels show similar average CSAT, while email has a visibly lower average score. Email support may need deeper review around turnaround time, resolution quality, communication clarity, or escalation handling.


## Visualization 5: Channel-wise CSAT Score Mix


In [ ]:
plt.close("all")
# Percentage distribution of each CSAT score within each channel
channel_score_mix = pd.crosstab(
    df["channel_name"],
    df["CSAT Score"],
    normalize="index"
).mul(100)

ax = sns.heatmap(channel_score_mix, annot=True, fmt=".1f", cmap="YlGnBu", linewidths=0.5)
ax.set_title("CSAT Score Mix by Channel (%)")
ax.set_xlabel("CSAT Score")
ax.set_ylabel("Channel")

plt.tight_layout()
plt.show()


**Business Insight:** The heatmap shows whether channel performance differences are caused by fewer top ratings or more low ratings. Email should be inspected for a comparatively weaker mix of score 5 responses and higher low-score exposure.


# 9. Issue and Product Category Analysis


## Visualization 6: Issue Category vs Average CSAT Score


In [ ]:
plt.close("all")
# Average CSAT by issue category
category_csat = (
    df.groupby("category", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
    .sort_values("avg_csat", ascending=True)
)

ax = sns.barplot(data=category_csat, y="category", x="avg_csat", palette="crest")
ax.set_title("Average CSAT Score by Issue Category")
ax.set_xlabel("Average CSAT Score")
ax.set_ylabel("Issue Category")
ax.set_xlim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
category_csat


**Business Insight:** Issue categories do not perform equally. Cancellation, product queries, and lower-volume miscellaneous categories show weaker CSAT than top-performing categories. These segments are good candidates for better self-service content, clearer policies, and agent coaching.


## Visualization 7: Product Category vs Average CSAT Score


In [ ]:
plt.close("all")
# Product category analysis uses only records where product category is available.
product_df = df.dropna(subset=["Product_category"])

product_csat = (
    product_df.groupby("Product_category", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
    .sort_values("avg_csat", ascending=True)
)

ax = sns.barplot(data=product_csat, y="Product_category", x="avg_csat", palette="rocket")
ax.set_title("Average CSAT Score by Product Category")
ax.set_xlabel("Average CSAT Score")
ax.set_ylabel("Product Category")
ax.set_xlim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
product_csat


**Business Insight:** Product categories with lower average CSAT, such as mobile, home appliances, furniture, or gift card-related interactions, may have more complex resolution journeys. Because product category is missing for many records, the finding should guide targeted investigation rather than final ranking alone.


## Visualization 8: Product Category Contact Volume


In [ ]:
plt.close("all")
# Product category volume among available product-category records
product_volume = product_df["Product_category"].value_counts().reset_index()
product_volume.columns = ["Product_category", "interactions"]

ax = sns.barplot(data=product_volume, y="Product_category", x="interactions", palette="flare")
ax.set_title("Interaction Volume by Product Category")
ax.set_xlabel("Number of Interactions")
ax.set_ylabel("Product Category")

for container in ax.containers:
    ax.bar_label(container, fmt="%d", padding=3)

plt.tight_layout()
plt.show()


**Business Insight:** Electronics, lifestyle, and books/general merchandise drive most product-tagged support volume. Improving support playbooks for high-volume product categories can create broader CSAT impact than focusing only on small niche segments.


# 10. Handling Time and Response Time Analysis


## Visualization 9: Connected Handling Time vs CSAT Score


In [ ]:
plt.close("all")
# Connected handling time is very sparse, so analyze only available records.
handling_df = df.dropna(subset=["connected_handling_time", "CSAT Score"]).copy()

ax = sns.boxplot(data=handling_df, x="CSAT Score", y="connected_handling_time", palette="coolwarm")
ax.set_title("Connected Handling Time Distribution by CSAT Score")
ax.set_xlabel("CSAT Score")
ax.set_ylabel("Connected Handling Time")

plt.tight_layout()
plt.show()

print(f"Records with connected handling time: {len(handling_df):,}")
print(f"Correlation with CSAT: {handling_df[['connected_handling_time', 'CSAT Score']].corr().iloc[0, 1]:.3f}")


**Business Insight:** Connected handling time is available for only a small subset of records, so this chart should be read cautiously. In the available subset, handling time does not show a strong linear relationship with CSAT, suggesting that resolution quality may matter more than duration alone.


## Visualization 10: Response Time Buckets vs Average CSAT


In [ ]:
plt.close("all")
# Response time buckets using valid non-negative response times
response_df = df.dropna(subset=["valid_response_time_minutes"]).copy()
response_df["response_time_bucket"] = pd.cut(
    response_df["valid_response_time_minutes"],
    bins=[0, 5, 15, 60, 240, np.inf],
    labels=["0-5 min", "5-15 min", "15-60 min", "1-4 hrs", ">4 hrs"],
    include_lowest=True
)

response_bucket_csat = (
    response_df.groupby("response_time_bucket", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
)

ax = sns.barplot(data=response_bucket_csat, x="response_time_bucket", y="avg_csat", palette="cubehelix")
ax.set_title("Average CSAT by Response Time Bucket")
ax.set_xlabel("Response Time Bucket")
ax.set_ylabel("Average CSAT Score")
ax.set_ylim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
response_bucket_csat


**Business Insight:** Faster responses generally protect customer satisfaction. Longer response-time buckets should be monitored for process bottlenecks, especially if they combine high volume with below-average CSAT.


# 11. Agent Performance Analysis


## Visualization 11: Top and Bottom Agent Performance by Average CSAT


In [ ]:
plt.close("all")
# Agent performance with a minimum interaction threshold for fairness.
min_interactions = 30
agent_perf = (
    df.groupby("Agent_name", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
    .query("interactions >= @min_interactions")
)

top_agents = agent_perf.sort_values("avg_csat", ascending=False).head(10)
bottom_agents = agent_perf.sort_values("avg_csat", ascending=True).head(10)
agent_compare = pd.concat([
    top_agents.assign(group="Top 10"),
    bottom_agents.assign(group="Bottom 10")
])

plt.figure(figsize=(12, 8))
ax = sns.barplot(data=agent_compare, y="Agent_name", x="avg_csat", hue="group", dodge=False, palette=["#2a9d8f", "#e76f51"])
ax.set_title(f"Top and Bottom Agents by Average CSAT (Minimum {min_interactions} Interactions)")
ax.set_xlabel("Average CSAT Score")
ax.set_ylabel("Agent Name")
ax.set_xlim(0, 5)
ax.legend(title="Performance Group")

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
agent_compare.sort_values(["group", "avg_csat"], ascending=[False, False])


**Business Insight:** Agent-level CSAT varies substantially even after applying a minimum interaction threshold. High performers can be studied for best practices, while low performers may need coaching, quality audits, or review of case mix before taking corrective action.


## Visualization 12: Agent Shift vs Average CSAT Score


In [ ]:
plt.close("all")
# Shift-level performance
shift_csat = (
    df.groupby("Agent Shift", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
    .sort_values("avg_csat", ascending=False)
)

ax = sns.barplot(data=shift_csat, x="Agent Shift", y="avg_csat", palette="Spectral")
ax.set_title("Average CSAT Score by Agent Shift")
ax.set_xlabel("Agent Shift")
ax.set_ylabel("Average CSAT Score")
ax.set_ylim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
shift_csat


**Business Insight:** CSAT differs by shift, with some lower-volume shifts performing better than the highest-volume shifts. The morning shift carries the largest workload and should be reviewed for staffing, queue pressure, and quality consistency.


## Visualization 13: Agent Tenure Bucket vs Average CSAT Score


In [ ]:
plt.close("all")
# Tenure-level performance
tenure_order = ["0-30", "31-60", "61-90", ">90", "On Job Training"]
tenure_csat = (
    df.groupby("Tenure Bucket", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
)
tenure_csat["Tenure Bucket"] = pd.Categorical(tenure_csat["Tenure Bucket"], categories=tenure_order, ordered=True)
tenure_csat = tenure_csat.sort_values("Tenure Bucket")

ax = sns.barplot(data=tenure_csat, x="Tenure Bucket", y="avg_csat", palette="Set1")
ax.set_title("Average CSAT Score by Agent Tenure Bucket")
ax.set_xlabel("Tenure Bucket")
ax.set_ylabel("Average CSAT Score")
ax.set_ylim(0, 5)

for container in ax.containers:
    ax.bar_label(container, fmt="%.2f", padding=3)

plt.tight_layout()
plt.show()
tenure_csat


**Business Insight:** Agents in active on-the-job training show lower average CSAT than more experienced tenure buckets. This supports investment in onboarding quality, supervised case handling, and early-stage coaching.


# 12. Customer City Trends


## Visualization 14: Top Customer Cities by Volume and Average CSAT


In [ ]:
plt.close("all")
# City analysis uses only records where customer city is available.
city_df = df.dropna(subset=["Customer_City"]).copy()

city_summary = (
    city_df.groupby("Customer_City", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", interactions="count")
    .sort_values("interactions", ascending=False)
    .head(15)
)

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(data=city_summary, y="Customer_City", x="interactions", ax=axes[0], palette="Blues_r")
axes[0].set_title("Top 15 Cities by Interaction Volume")
axes[0].set_xlabel("Number of Interactions")
axes[0].set_ylabel("Customer City")

sns.barplot(data=city_summary.sort_values("avg_csat", ascending=True), y="Customer_City", x="avg_csat", ax=axes[1], palette="Greens_r")
axes[1].set_title("Average CSAT for Top 15 Cities")
axes[1].set_xlabel("Average CSAT Score")
axes[1].set_ylabel("")
axes[1].set_xlim(0, 5)

for axis in axes:
    for container in axis.containers:
        axis.bar_label(container, fmt="%.2f" if axis == axes[1] else "%d", padding=3)

plt.tight_layout()
plt.show()
city_summary


**Business Insight:** Major cities such as Hyderabad, New Delhi, Pune, Mumbai, and Bangalore contribute high support volume, but their average CSAT is not always the highest. City-level performance may reflect delivery, returns pickup, language, or regional service-partner differences and should be paired with operational geography data.


# 13. Time Trend Analysis


## Visualization 15: Daily CSAT Trend


In [ ]:
plt.close("all")
# Daily CSAT trend across the survey period
daily_csat = (
    df.dropna(subset=["Survey_response_Date"])
    .groupby("Survey_response_Date", as_index=False)["CSAT Score"]
    .agg(avg_csat="mean", responses="count")
)

daily_csat["date_label"] = daily_csat["Survey_response_Date"].dt.strftime("%d-%b")

fig, ax1 = plt.subplots(figsize=(13, 6))
ax2 = ax1.twinx()

sns.barplot(data=daily_csat, x="date_label", y="responses", ax=ax2, color="#d9d9d9", alpha=0.45)
sns.lineplot(data=daily_csat, x="date_label", y="avg_csat", marker="o", ax=ax1, color="#1f77b4")

ax1.set_title("Daily Average CSAT Trend")
ax1.set_xlabel("Survey Response Date")
ax1.set_ylabel("Average CSAT Score", color="#1f77b4")
ax1.set_ylim(0, 5)
ax1.tick_params(axis="x", rotation=45)
ax2.set_ylabel("Number of Responses", color="#666666")

plt.tight_layout()
plt.show()
daily_csat.head()


**Business Insight:** Daily CSAT trend monitoring helps detect operational incidents or process changes that may not be visible in the overall average. Since the dataset covers August 2023, this should be extended with more months for seasonality and campaign impact analysis.


# 14. Correlation View for Numeric Features


## Visualization 16: Numeric Correlation Heatmap


In [ ]:
plt.close("all")
# Numeric correlation view
numeric_cols = [
    "CSAT Score",
    "Item_price",
    "connected_handling_time",
    "response_time_minutes",
    "valid_response_time_minutes"
]
available_numeric_cols = [col for col in numeric_cols if col in df.columns]

corr_matrix = df[available_numeric_cols].corr()

ax = sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="vlag", center=0, linewidths=0.5)
ax.set_title("Correlation Heatmap of Numeric Features")

plt.tight_layout()
plt.show()


**Business Insight:** Numeric variables show limited direct correlation with CSAT, which suggests satisfaction is likely influenced by categorical and process factors such as issue type, channel, agent behavior, and resolution quality. Correlation should not be treated as causation, especially where missingness is high.


# 15. Key Findings and Recommendations

## Key Findings

- Overall CSAT is positive, with score 5 dominating the response distribution.
- Low CSAT responses are still meaningful in volume and should be actively monitored.
- Inbound support has the highest volume, so improvements there have the largest scale impact.
- Email has lower average CSAT than inbound and outcall, making it an important improvement area.
- Product and city analyses are useful but limited by high missingness.
- Connected handling time is too sparse for strong conclusions and should be improved as a data capture field.
- Agent-level performance varies materially, even after filtering for agents with sufficient interaction volume.
- On-the-job training agents have lower CSAT, indicating an onboarding and coaching opportunity.

## Business Recommendations

- Prioritize root-cause analysis for score 1 and score 2 interactions.
- Improve email support SLAs, response templates, escalation paths, and closure quality.
- Build targeted playbooks for lower-CSAT product categories and issue categories.
- Use high-performing agents as benchmarks for coaching and quality calibration.
- Strengthen onboarding support for agents in training.
- Improve data completeness for city, product category, item price, and handling time to support more reliable modeling.
- Track CSAT by day, channel, category, and agent as a recurring operations dashboard.


# 16. Conclusion

The Flipkart customer support dataset shows generally strong customer satisfaction, but there are clear opportunities to improve low-CSAT experiences. The most actionable areas are email support quality, issue-specific process improvements, product-category support playbooks, agent coaching, onboarding, and data quality enhancement. This EDA establishes a strong foundation for later predictive modeling or operational dashboard development.


# 17. ML Preparation Pipeline

This section prepares the EDA dataset for a downstream machine learning notebook. It does not replace or modify the previous EDA analysis. The goal is to create a reproducible, ML-ready dataframe named `df_clean`, define `X` and `y`, remove leakage-prone fields, handle missing and invalid values, engineer useful features, and encode categorical variables safely.


## 17.1 Prediction Problem Selection

The most appropriate modeling family for this dataset is **classification**, not regression.

`CSAT Score` is an ordinal customer rating from 1 to 5, and the score distribution is heavily concentrated at score 5. A regression model would treat the distance from 1 to 2 as equivalent to the distance from 4 to 5, even though the business meaning is different. For customer support operations, the more useful objective is to identify satisfaction-risk groups that can trigger intervention.

As an initial benchmark, this notebook creates a three-class classification label:

- `Low`: CSAT scores 1 and 2
- `Neutral`: CSAT score 3
- `High`: CSAT scores 4 and 5

This 3-class target is useful for exploratory model comparison because it preserves the ordinal business interpretation of CSAT. However, because the `Neutral` class is very small, Section 19 explicitly compares this formulation against binary alternatives and makes the final production target recommendation.


In [ ]:
# Create a separate ML preparation dataframe from the EDA dataframe.
df_clean = df.copy()

# Classification target based on business-friendly CSAT categories.
df_clean["csat_target"] = pd.cut(
    df_clean["CSAT Score"],
    bins=[0, 2, 3, 5],
    labels=["Low", "Neutral", "High"],
    include_lowest=True
)

# Confirm target distribution.
target_distribution = pd.DataFrame({
    "count": df_clean["csat_target"].value_counts(),
    "percentage": (df_clean["csat_target"].value_counts(normalize=True) * 100).round(2)
})
target_distribution


## 17.2 Missingness Analysis

Before imputing or dropping columns, missingness is reviewed at the column level. Columns with extreme missingness can add noise and instability to ML models. Sparse columns are dropped only when the missing rate is too high for reliable imputation or when the column is not expected to provide predictive value in its current form.


In [ ]:
# Missingness analysis for ML preparation.
ml_missing_summary = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_percent": (df_clean.isna().mean() * 100).round(2),
    "dtype": df_clean.dtypes.astype(str)
}).sort_values("missing_percent", ascending=False)

ml_missing_summary


## 17.3 Handle Invalid and Impossible Values

Negative response times are not logically valid because a support response cannot happen before the issue is reported. These records are treated as invalid for the engineered response-time feature and replaced with missing values, allowing the imputation step to handle them consistently.

For `Item_price`, negative values would be impossible in this context. If such values exist, they are also replaced with missing values before imputation.


In [ ]:
# Ensure response time exists even if this section is run independently after loading the dataset.
if "response_time_minutes" not in df_clean.columns:
    df_clean["Issue_reported at"] = pd.to_datetime(df_clean["Issue_reported at"], errors="coerce", dayfirst=True)
    df_clean["issue_responded"] = pd.to_datetime(df_clean["issue_responded"], errors="coerce", dayfirst=True)
    df_clean["response_time_minutes"] = (
        df_clean["issue_responded"] - df_clean["Issue_reported at"]
    ).dt.total_seconds() / 60

# Negative response times are invalid; mark them as missing for later imputation.
invalid_response_time_count = (df_clean["response_time_minutes"] < 0).sum()
df_clean.loc[df_clean["response_time_minutes"] < 0, "response_time_minutes"] = np.nan

# Negative item prices are impossible; mark them as missing if present.
invalid_item_price_count = 0
if "Item_price" in df_clean.columns:
    invalid_item_price_count = (df_clean["Item_price"] < 0).sum()
    df_clean.loc[df_clean["Item_price"] < 0, "Item_price"] = np.nan

print(f"Invalid negative response times corrected to missing: {invalid_response_time_count:,}")
print(f"Invalid negative item prices corrected to missing: {invalid_item_price_count:,}")


## 17.4 ML Feature Engineering

The original timestamps are converted into model-friendly features. These features capture operational timing patterns without exposing raw timestamp strings to the model.

Engineered features include:

- `response_time_minutes`
- `response_time_bucket`
- `issue_hour`
- `issue_weekday`
- `survey_weekday`
- `is_weekend_issue`
- `same_day_response`
- `response_delay_flag`
- `has_customer_remark`
- `has_order_id`
- `agent_interaction_count`
- `agent_historical_avg_csat`
- `supervisor_interaction_count`
- `manager_interaction_count`

Historical aggregate features are computed with the current full dataset for ML preparation only. In a formal train/test modeling workflow, these should be computed inside cross-validation or on training folds only to avoid leakage.


In [ ]:
# Parse date/time columns if needed.
df_clean["Issue_reported at"] = pd.to_datetime(df_clean["Issue_reported at"], errors="coerce", dayfirst=True)
df_clean["issue_responded"] = pd.to_datetime(df_clean["issue_responded"], errors="coerce", dayfirst=True)
df_clean["Survey_response_Date"] = pd.to_datetime(df_clean["Survey_response_Date"], errors="coerce", dayfirst=True)

# Time-based features.
df_clean["issue_hour"] = df_clean["Issue_reported at"].dt.hour
df_clean["issue_weekday"] = df_clean["Issue_reported at"].dt.day_name()
df_clean["survey_weekday"] = df_clean["Survey_response_Date"].dt.day_name()
df_clean["is_weekend_issue"] = df_clean["Issue_reported at"].dt.dayofweek.isin([5, 6]).astype("int")

# Response time features.
df_clean["same_day_response"] = (
    df_clean["Issue_reported at"].dt.date == df_clean["issue_responded"].dt.date
).astype("int")

df_clean["response_delay_flag"] = np.select(
    [
        df_clean["response_time_minutes"].isna(),
        df_clean["response_time_minutes"] <= 15,
        df_clean["response_time_minutes"] <= 60,
        df_clean["response_time_minutes"] <= 240,
        df_clean["response_time_minutes"] > 240
    ],
    ["Unknown", "Fast", "Moderate", "Delayed", "Severely Delayed"],
    default="Unknown"
)

df_clean["response_time_bucket"] = pd.cut(
    df_clean["response_time_minutes"],
    bins=[0, 5, 15, 60, 240, np.inf],
    labels=["0-5 min", "5-15 min", "15-60 min", "1-4 hrs", ">4 hrs"],
    include_lowest=True
).astype("object")
df_clean["response_time_bucket"] = df_clean["response_time_bucket"].fillna("Unknown")

# Business presence flags.
df_clean["has_customer_remark"] = df_clean["Customer Remarks"].notna().astype("int")
df_clean["has_order_id"] = df_clean["Order_id"].notna().astype("int")

# High-cardinality entity summaries.
# These summarize operational exposure without one-hot encoding thousands of names/cities.
df_clean["agent_interaction_count"] = df_clean.groupby("Agent_name")["Agent_name"].transform("count")
df_clean["agent_historical_avg_csat"] = df_clean.groupby("Agent_name")["CSAT Score"].transform("mean")
df_clean["supervisor_interaction_count"] = df_clean.groupby("Supervisor")["Supervisor"].transform("count")
df_clean["manager_interaction_count"] = df_clean.groupby("Manager")["Manager"].transform("count")

# City volume feature while avoiding high-cardinality one-hot encoding.
df_clean["city_interaction_count"] = df_clean.groupby("Customer_City")["Customer_City"].transform("count")
df_clean["city_interaction_count"] = df_clean["city_interaction_count"].fillna(0)

df_clean[[
    "response_time_minutes", "response_time_bucket", "issue_hour", "issue_weekday",
    "survey_weekday", "response_delay_flag", "has_customer_remark",
    "agent_interaction_count", "agent_historical_avg_csat"
]].head()


## 17.5 Remove Leakage-Prone, Identifier, and Excessively Sparse Columns

The following fields are removed before modeling:

- `Unique id` and `Order_id`: row/order identifiers with no generalizable predictive meaning.
- Raw timestamp columns: converted into ML-ready date/time features.
- `Customer Remarks`: free-text field requiring separate NLP handling; replaced here by `has_customer_remark`.
- `CSAT Score` and `CSAT_Group`: source target fields that would leak the classification target.
- `connected_handling_time`: dropped because it is missing for almost the entire dataset.
- `order_date_time`: dropped due to very high missingness and because issue-response timing is already represented.
- `Customer_City`: high-cardinality and highly missing; represented by `city_interaction_count` instead.
- `Agent_name`, `Supervisor`, `Manager`: high-cardinality operational identifiers; represented through aggregate count features instead.

This keeps the feature set more stable and reduces the risk of memorizing individual IDs or post-outcome information.


In [ ]:
# Columns removed from model features.
leakage_or_identifier_cols = [
    "Unique id",
    "Order_id",
    "Customer Remarks",
    "Issue_reported at",
    "issue_responded",
    "Survey_response_Date",
    "CSAT Score",
    "CSAT_Group"
]

sparse_cols_to_drop = [
    "connected_handling_time",
    "order_date_time"
]

high_cardinality_cols_to_drop = [
    "Customer_City",
    "Agent_name",
    "Supervisor",
    "Manager"
]

columns_to_remove = [
    col for col in leakage_or_identifier_cols + sparse_cols_to_drop + high_cardinality_cols_to_drop
    if col in df_clean.columns
]

removed_features_summary = pd.DataFrame({
    "removed_column": columns_to_remove,
    "reason": [
        "identifier/leakage/raw text/raw timestamp/target" if col in leakage_or_identifier_cols
        else "excessively sparse"
        if col in sparse_cols_to_drop
        else "high-cardinality field represented by aggregate feature"
        for col in columns_to_remove
    ]
})

removed_features_summary


## 17.6 Define Raw Feature Matrix and Target

`y` is the classification target. `X_raw` contains the selected model inputs before imputation and encoding. The final encoded feature dataframe is created after missing-value handling and categorical encoding.


In [ ]:
# Define target.
y = df_clean["csat_target"].astype("category")

# Remove target and excluded columns from raw feature matrix.
X_raw = df_clean.drop(columns=columns_to_remove + ["csat_target"], errors="ignore")

print(f"Raw feature matrix shape before encoding: {X_raw.shape}")
print(f"Target shape: {y.shape}")
X_raw.head()


## 17.7 Missing Value Imputation

Numerical features are imputed using the median because it is robust to outliers such as long response times and high item prices. Categorical features are imputed using the most frequent category because these fields represent business labels where the mode is a stable and interpretable default.

Any remaining missing response-time bucket values are already assigned to `Unknown`, preserving missingness as a useful signal.


In [ ]:
# Separate numeric and categorical features.
numeric_features = X_raw.select_dtypes(include=["number", "bool"]).columns.tolist()
categorical_features = X_raw.select_dtypes(exclude=["number", "bool"]).columns.tolist()

# Impute numerical columns with median.
X_numeric = X_raw[numeric_features].copy()
for col in numeric_features:
    median_value = X_numeric[col].median()
    X_numeric[col] = X_numeric[col].fillna(median_value)

# Impute categorical columns with mode.
X_categorical = X_raw[categorical_features].copy()
for col in categorical_features:
    mode_series = X_categorical[col].mode(dropna=True)
    fill_value = mode_series.iloc[0] if not mode_series.empty else "Unknown"
    X_categorical[col] = X_categorical[col].fillna(fill_value).astype(str)

imputation_summary = pd.DataFrame({
    "feature_type": ["numeric", "categorical"],
    "feature_count": [len(numeric_features), len(categorical_features)],
    "imputation_strategy": ["median", "most frequent category"]
})

imputation_summary


## 17.8 Categorical Encoding

Low-cardinality categorical features are encoded using `OneHotEncoder` with `handle_unknown='ignore'`. This is appropriate because these features have a manageable number of categories and the model may encounter unseen categories in future data.

High-cardinality fields such as agent, supervisor, manager, and city were not one-hot encoded. They were replaced by aggregate count or historical features to avoid creating thousands of sparse columns and to reduce memorization risk.


In [ ]:
# One-hot encode selected categorical features.
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

if categorical_features:
    encoded_array = encoder.fit_transform(X_categorical)
    encoded_feature_names = encoder.get_feature_names_out(categorical_features)
    X_encoded_categorical = pd.DataFrame(
        encoded_array,
        columns=encoded_feature_names,
        index=X_categorical.index
    )
else:
    X_encoded_categorical = pd.DataFrame(index=X_raw.index)

# Combine imputed numeric features and encoded categorical features.
X = pd.concat([X_numeric, X_encoded_categorical], axis=1)

# Final ML-ready dataframe includes encoded features plus target.
df_clean = X.copy()
df_clean["csat_target"] = y.values

print(f"Encoded feature matrix shape: {X.shape}")
print(f"Final df_clean shape: {df_clean.shape}")
df_clean.head()


## 17.9 Final Validation Checks

The final ML-ready dataframe should have no missing feature values, a clearly defined classification target, and only numeric encoded features plus the target label.


In [ ]:
# Validate the final ML-ready dataset.
feature_missing_count = X.isna().sum().sum()
target_missing_count = pd.Series(y).isna().sum()
non_numeric_feature_count = X.select_dtypes(exclude=["number", "bool"]).shape[1]

print(f"Missing values in X: {feature_missing_count:,}")
print(f"Missing values in y: {target_missing_count:,}")
print(f"Non-numeric feature columns in X: {non_numeric_feature_count:,}")
print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print(f"df_clean shape: {df_clean.shape}")


## 17.10 Final ML Preparation Summary


In [ ]:
# Concise ML preparation summary.
features_retained = X.columns.tolist()
features_removed = removed_features_summary["removed_column"].tolist()

df_clean_summary = pd.DataFrame({
    "Item": [
        "Prediction problem",
        "Target variable",
        "Target classes",
        "Number of retained encoded features",
        "Number of removed original columns",
        "Final df_clean rows",
        "Final df_clean columns"
    ],
    "Value": [
        "Multi-class classification",
        "csat_target",
        ", ".join(y.cat.categories.astype(str)),
        len(features_retained),
        len(features_removed),
        df_clean.shape[0],
        df_clean.shape[1]
    ]
})

print("Features removed:")
print(features_removed)
print("\nFirst 25 retained encoded features:")
print(features_retained[:25])

df_clean_summary


### ML-Ready Output

At the end of this section:

- `df_clean` contains the final ML-ready encoded dataframe.
- `X` contains the encoded feature matrix.
- `y` contains the classification target `csat_target`.
- Leakage-prone identifiers and sparse columns have been removed.
- Numeric and categorical missing values have been imputed.
- Low-cardinality categorical features have been one-hot encoded.
- High-cardinality identifiers have been handled through aggregate features instead of direct one-hot encoding.


# 18. Production-Style Machine Learning Benchmark

This section uses the already prepared `X` and `y` objects to compare multiple classification models, handle class imbalance, select the best model, tune it with stratified cross-validation, and generate explainability outputs. It should be read as a **production-style benchmark for the 3-class target**, not the final deployment decision.

The workflow is designed for a high-quality intern evaluation submission: reproducible, metric-driven, leakage-aware, and business-oriented. The final target formulation is revisited in Section 19 because model usefulness depends on both predictive performance and operational actionability.


## 18.1 Modeling Setup and Leakage Control

Although `X` and `y` were prepared in the previous section, a production workflow must still verify leakage risk before modeling. The feature `agent_historical_avg_csat` is derived directly from the target (`CSAT Score`) using the full dataset. This can inflate model performance because it allows the model to learn historical satisfaction outcomes computed from the same labels it is trying to predict.

Therefore, `agent_historical_avg_csat` is removed from the modeling matrix. The remaining aggregate count features are retained because they describe workload or exposure, not the target outcome itself.


In [ ]:
# Production modeling setup.
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.inspection import permutation_importance
from sklearn.multiclass import OneVsRestClassifier
from scipy.stats import randint, uniform
import time

RANDOM_STATE = 42

# Remove target-derived full-data aggregate feature to avoid leakage.
leakage_features_for_modeling = ["agent_historical_avg_csat"]
X_model = X.drop(columns=[col for col in leakage_features_for_modeling if col in X.columns], errors="ignore").copy()
y_model = y.astype(str).copy()
class_labels = sorted(y_model.unique())

print(f"Original X shape: {X.shape}")
print(f"Leakage-aware X_model shape: {X_model.shape}")
print(f"Removed leakage features: {[col for col in leakage_features_for_modeling if col in X.columns]}")
print(f"Target classes: {class_labels}")


## 18.2 Stratified Train-Test Split

A stratified split is required because the target classes are imbalanced. Stratification preserves the proportion of `Low`, `Neutral`, and `High` CSAT classes in both training and test sets, making evaluation more reliable and comparable across models.


In [ ]:
# Stratified train-test split.
X_train, X_test, y_train, y_test = train_test_split(
    X_model,
    y_model,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y_model
)

split_summary = pd.DataFrame({
    "dataset": ["Train", "Test"],
    "rows": [X_train.shape[0], X_test.shape[0]],
    "features": [X_train.shape[1], X_test.shape[1]]
})

split_summary


## 18.3 Class Imbalance Analysis and Evaluation Strategy

The class distribution is skewed toward `High` CSAT. In this setting, accuracy alone can be misleading because a model can look strong by mostly predicting the majority class. To evaluate interns fairly on model quality, the primary ranking metric is **Macro F1**, which gives equal weight to each class regardless of class size.

The evaluation also reports:

- **Balanced Accuracy**: recall averaged across classes.
- **Macro Precision / Recall / F1**: equal class weighting.
- **Weighted F1**: class-size-weighted performance.
- **Classification Report**: detailed per-class precision, recall, and F1.
- **Confusion Matrix**: error pattern visibility.


In [ ]:
# Quantify class imbalance.
class_distribution = pd.DataFrame({
    "count": y_model.value_counts(),
    "percentage": (y_model.value_counts(normalize=True) * 100).round(2)
}).rename_axis("csat_target").reset_index()

majority_class_pct = class_distribution["percentage"].max()
minority_class_pct = class_distribution["percentage"].min()
imbalance_ratio = class_distribution["count"].max() / class_distribution["count"].min()

print(f"Majority class percentage: {majority_class_pct:.2f}%")
print(f"Minority class percentage: {minority_class_pct:.2f}%")
print(f"Imbalance ratio: {imbalance_ratio:.2f}:1")

class_distribution


## 18.4 Model Definitions

The workflow compares linear, tree-based, bagging, and boosting classifiers. Class imbalance is handled through `class_weight='balanced'` where the algorithm supports it. For Gradient Boosting, which does not support `class_weight`, balanced `sample_weight` is passed during fitting.

XGBoost and LightGBM are included if the libraries are available in the runtime. If they are not installed, the notebook reports that they were skipped without failing.


In [ ]:
# Optional model imports.
optional_model_status = []

try:
    from xgboost import XGBClassifier
    XGBOOST_AVAILABLE = True
    optional_model_status.append({"model": "XGBoost", "status": "Available"})
except Exception as exc:
    XGBOOST_AVAILABLE = False
    optional_model_status.append({"model": "XGBoost", "status": f"Skipped - {type(exc).__name__}: {exc}"})

try:
    from lightgbm import LGBMClassifier
    LIGHTGBM_AVAILABLE = True
    optional_model_status.append({"model": "LightGBM", "status": "Available"})
except Exception as exc:
    LIGHTGBM_AVAILABLE = False
    optional_model_status.append({"model": "LightGBM", "status": f"Skipped - {type(exc).__name__}: {exc}"})

optional_model_status_df = pd.DataFrame(optional_model_status)
optional_model_status_df


In [ ]:
# Define models for comparison.
models = {
    "Logistic Regression": {
        "estimator": make_pipeline(
            StandardScaler(),
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE
            )
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False
    },
    "Decision Tree": {
        "estimator": DecisionTreeClassifier(
            class_weight="balanced",
            max_depth=12,
            min_samples_leaf=30,
            random_state=RANDOM_STATE
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False
    },
    "Random Forest": {
        "estimator": RandomForestClassifier(
            n_estimators=80,
            class_weight="balanced_subsample",
            max_depth=18,
            min_samples_leaf=10,
            n_jobs=1,
            random_state=RANDOM_STATE
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False
    },
    "Gradient Boosting": {
        "estimator": GradientBoostingClassifier(
            n_estimators=60,
            learning_rate=0.08,
            max_depth=3,
            random_state=RANDOM_STATE
        ),
        "uses_sample_weight": True,
        "requires_encoded_target": False
    }
}

# Label encoding is prepared for optional libraries that require numeric targets.
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

if XGBOOST_AVAILABLE:
    models["XGBoost"] = {
        "estimator": XGBClassifier(
            n_estimators=80,
            max_depth=4,
            learning_rate=0.08,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1
        ),
        "uses_sample_weight": True,
        "requires_encoded_target": True
    }

if LIGHTGBM_AVAILABLE:
    models["LightGBM"] = {
        "estimator": LGBMClassifier(
            n_estimators=100,
            learning_rate=0.06,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbose=-1
        ),
        "uses_sample_weight": False,
        "requires_encoded_target": False
    }

list(models.keys())


## 18.5 Train and Evaluate Baseline Models

Every model is trained on the same stratified training split and evaluated on the same held-out test split. The comparison table is ranked by Macro F1 first and Weighted F1 second.


In [ ]:
# Helper function for model evaluation.
def evaluate_predictions(model_name, y_true, y_pred):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Balanced Accuracy": balanced_accuracy_score(y_true, y_pred),
        "Macro Precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_true, y_pred, average="weighted", zero_division=0)
    }

# Train and evaluate models.
model_results = []
trained_models = {}
classification_reports = {}
confusion_matrices = {}
training_times = {}

sample_weights_train = compute_sample_weight(class_weight="balanced", y=y_train)
sample_weights_train_encoded = compute_sample_weight(class_weight="balanced", y=y_train_encoded)

for model_name, config in models.items():
    estimator = config["estimator"]
    start_time = time.time()

    if config["requires_encoded_target"]:
        fit_y = y_train_encoded
        if config["uses_sample_weight"]:
            estimator.fit(X_train, fit_y, sample_weight=sample_weights_train_encoded)
        else:
            estimator.fit(X_train, fit_y)
        y_pred_encoded = estimator.predict(X_test)
        y_pred = label_encoder.inverse_transform(y_pred_encoded.astype(int))
    else:
        if config["uses_sample_weight"]:
            estimator.fit(X_train, y_train, sample_weight=sample_weights_train)
        else:
            estimator.fit(X_train, y_train)
        y_pred = estimator.predict(X_test)

    elapsed_time = time.time() - start_time
    trained_models[model_name] = estimator
    training_times[model_name] = elapsed_time

    metrics = evaluate_predictions(model_name, y_test, y_pred)
    metrics["Training Time Seconds"] = round(elapsed_time, 2)
    model_results.append(metrics)

    classification_reports[model_name] = classification_report(
        y_test,
        y_pred,
        labels=class_labels,
        zero_division=0,
        output_dict=True
    )
    confusion_matrices[model_name] = confusion_matrix(y_test, y_pred, labels=class_labels)

comparison_table = (
    pd.DataFrame(model_results)
    .sort_values(["Macro F1", "Weighted F1"], ascending=False)
    .reset_index(drop=True)
)

comparison_table


## 18.6 Classification Reports and Confusion Matrices


In [ ]:
# Display classification reports for each model.
for model_name in comparison_table["Model"]:
    print("=" * 90)
    print(f"Classification Report: {model_name}")
    print("=" * 90)
    report_df = pd.DataFrame(classification_reports[model_name]).T
    display(report_df.round(3))


In [ ]:
plt.close("all")
# Plot confusion matrices for all compared models.
num_models = len(comparison_table)
cols = 2
rows = int(np.ceil(num_models / cols))
fig, axes = plt.subplots(rows, cols, figsize=(12, 5 * rows))
axes = np.array(axes).reshape(-1)

for ax, model_name in zip(axes, comparison_table["Model"]):
    disp = ConfusionMatrixDisplay(
        confusion_matrix=confusion_matrices[model_name],
        display_labels=class_labels
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"Confusion Matrix: {model_name}")

for ax in axes[num_models:]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## 18.7 Best Model Selection

The best model is selected using Macro F1 as the primary metric and Weighted F1 as the secondary metric. Macro F1 is preferred because it prevents the majority `High` CSAT class from dominating the selection decision.


In [ ]:
# Select best baseline model.
best_model_name = comparison_table.loc[0, "Model"]
best_model = trained_models[best_model_name]
best_baseline_metrics = comparison_table.iloc[0].to_dict()

print(f"Best baseline model: {best_model_name}")
print("Selection rationale: highest Macro F1, with Weighted F1 used as tie-breaker.")
pd.DataFrame([best_baseline_metrics])


## 18.8 Hyperparameter Tuning with Stratified Cross-Validation

The selected model is tuned with stratified cross-validation. Stratification keeps class proportions stable across folds, which is important for the imbalanced target. The tuning search is intentionally compact enough to run reliably in a notebook while still testing meaningful parameter variation.


In [ ]:
# Build tuning configuration for the selected model.
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

def get_tuning_setup(model_name):
    if model_name == "Logistic Regression":
        estimator = make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1200, random_state=RANDOM_STATE)
        )
        param_distributions = {
            "logisticregression__C": uniform(0.05, 5.0),
            "logisticregression__solver": ["lbfgs", "liblinear"]
        }
        return estimator, param_distributions, False, False

    if model_name == "Decision Tree":
        estimator = DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE)
        param_distributions = {
            "max_depth": randint(4, 25),
            "min_samples_leaf": randint(10, 120),
            "min_samples_split": randint(20, 200),
            "criterion": ["gini", "entropy"]
        }
        return estimator, param_distributions, False, False

    if model_name == "Random Forest":
        estimator = RandomForestClassifier(class_weight="balanced_subsample", n_jobs=1, random_state=RANDOM_STATE)
        param_distributions = {
            "n_estimators": randint(60, 140),
            "max_depth": randint(8, 26),
            "min_samples_leaf": randint(5, 60),
            "max_features": ["sqrt", "log2", None]
        }
        return estimator, param_distributions, False, False

    if model_name == "Gradient Boosting":
        estimator = GradientBoostingClassifier(random_state=RANDOM_STATE)
        param_distributions = {
            "n_estimators": randint(40, 100),
            "learning_rate": uniform(0.03, 0.12),
            "max_depth": randint(2, 5),
            "min_samples_leaf": randint(10, 80)
        }
        return estimator, param_distributions, True, False

    if model_name == "XGBoost" and XGBOOST_AVAILABLE:
        estimator = XGBClassifier(
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1
        )
        param_distributions = {
            "n_estimators": randint(60, 140),
            "max_depth": randint(3, 7),
            "learning_rate": uniform(0.03, 0.12),
            "subsample": uniform(0.75, 0.25),
            "colsample_bytree": uniform(0.75, 0.25)
        }
        return estimator, param_distributions, True, True

    if model_name == "LightGBM" and LIGHTGBM_AVAILABLE:
        estimator = LGBMClassifier(class_weight="balanced", random_state=RANDOM_STATE, n_jobs=1, verbose=-1)
        param_distributions = {
            "n_estimators": randint(60, 160),
            "num_leaves": randint(15, 64),
            "learning_rate": uniform(0.03, 0.12),
            "min_child_samples": randint(10, 80)
        }
        return estimator, param_distributions, False, False

    raise ValueError(f"No tuning setup available for {model_name}")

tuning_estimator, param_distributions, tuning_uses_sample_weight, tuning_requires_encoded_target = get_tuning_setup(best_model_name)

search = RandomizedSearchCV(
    estimator=tuning_estimator,
    param_distributions=param_distributions,
    n_iter=6,
    scoring="f1_macro",
    cv=cv,
    random_state=RANDOM_STATE,
    n_jobs=1 if best_model_name != "Gradient Boosting" else 1,
    verbose=0
)

if tuning_requires_encoded_target:
    search.fit(
        X_train,
        y_train_encoded,
        sample_weight=sample_weights_train_encoded if tuning_uses_sample_weight else None
    )
else:
    if tuning_uses_sample_weight:
        search.fit(X_train, y_train, sample_weight=sample_weights_train)
    else:
        search.fit(X_train, y_train)

tuned_best_model = search.best_estimator_

print(f"Tuned model: {best_model_name}")
print(f"Best CV Macro F1: {search.best_score_:.4f}")
print("Best hyperparameters:")
print(search.best_params_)


## 18.9 Final Test Performance of Tuned Model


In [ ]:
# Evaluate tuned model on the held-out test set.
if tuning_requires_encoded_target:
    tuned_pred_encoded = tuned_best_model.predict(X_test)
    tuned_y_pred = label_encoder.inverse_transform(tuned_pred_encoded.astype(int))
else:
    tuned_y_pred = tuned_best_model.predict(X_test)

final_test_metrics = evaluate_predictions(f"Tuned {best_model_name}", y_test, tuned_y_pred)
final_test_metrics["Best CV Macro F1"] = search.best_score_

print("Final tuned model classification report:")
print(classification_report(y_test, tuned_y_pred, labels=class_labels, zero_division=0))

final_test_performance = pd.DataFrame([final_test_metrics])
final_test_performance


In [ ]:
plt.close("all")
# Confusion matrix for the tuned best model.
tuned_cm = confusion_matrix(y_test, tuned_y_pred, labels=class_labels)
ConfusionMatrixDisplay(tuned_cm, display_labels=class_labels).plot(cmap="Blues", values_format="d")
plt.title(f"Confusion Matrix: Tuned {best_model_name}")
plt.tight_layout()
plt.show()


## 18.10 Feature Importance

Feature importance is extracted directly from the tuned model when available. For linear models, absolute coefficient magnitude is used. For tree-based models, native feature importance is used.


In [ ]:
# Extract model-specific feature importance.
def get_feature_importance(estimator, feature_names):
    if hasattr(estimator, "feature_importances_"):
        importance_values = estimator.feature_importances_
        importance_type = "native_feature_importance"
    elif hasattr(estimator, "named_steps") and "logisticregression" in estimator.named_steps:
        logistic_model = estimator.named_steps["logisticregression"]
        importance_values = np.abs(logistic_model.coef_).mean(axis=0)
        importance_type = "mean_absolute_logistic_coefficient"
    elif hasattr(estimator, "coef_"):
        importance_values = np.abs(estimator.coef_).mean(axis=0)
        importance_type = "mean_absolute_coefficient"
    else:
        return pd.DataFrame(columns=["feature", "importance", "importance_type"])

    return (
        pd.DataFrame({
            "feature": feature_names,
            "importance": importance_values,
            "importance_type": importance_type
        })
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )

feature_importance_df = get_feature_importance(tuned_best_model, X_model.columns)
feature_importance_df.head(20)


In [ ]:
plt.close("all")
# Plot top feature importances when available.
if not feature_importance_df.empty:
    top_n = 20
    plot_df = feature_importance_df.head(top_n).sort_values("importance", ascending=True)
    ax = sns.barplot(data=plot_df, x="importance", y="feature", palette="viridis")
    ax.set_title(f"Top {top_n} Feature Importances - Tuned {best_model_name}")
    ax.set_xlabel("Importance")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("Native feature importance is not available for the selected model.")


## 18.11 Permutation Importance

Permutation importance measures how much the model performance drops when each feature is randomly shuffled. This is model-agnostic and is computed on a test-set sample for runtime efficiency.


In [ ]:
# Permutation importance on a test sample for computational efficiency.
permutation_sample_size = min(5000, X_test.shape[0])
X_perm = X_test.sample(n=permutation_sample_size, random_state=RANDOM_STATE)
y_perm = y_test.loc[X_perm.index]

permutation_result = permutation_importance(
    tuned_best_model,
    X_perm,
    y_perm,
    scoring="f1_macro",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1
)

permutation_importance_df = (
    pd.DataFrame({
        "feature": X_model.columns,
        "importance_mean": permutation_result.importances_mean,
        "importance_std": permutation_result.importances_std
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

permutation_importance_df.head(20)


In [ ]:
plt.close("all")
# Plot top permutation importances.
plot_df = permutation_importance_df.head(20).sort_values("importance_mean", ascending=True)
ax = sns.barplot(data=plot_df, x="importance_mean", y="feature", palette="mako")
ax.set_title(f"Top 20 Permutation Importances - Tuned {best_model_name}")
ax.set_xlabel("Mean Macro F1 Decrease")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.show()


## 18.12 SHAP Analysis

SHAP is attempted only if the `shap` package is installed and the selected model is compatible. If SHAP is unavailable or too expensive for the current runtime, the notebook continues with native and permutation importance outputs.


In [ ]:
# Optional SHAP analysis.
shap_status = "Not attempted"
shap_top_features = pd.DataFrame()

try:
    import shap
    shap_sample_size = min(1000, X_test.shape[0])
    X_shap = X_test.sample(n=shap_sample_size, random_state=RANDOM_STATE)

    # SHAP is most reliable here for tree-based estimators without a preprocessing pipeline.
    if hasattr(tuned_best_model, "feature_importances_"):
        explainer = shap.TreeExplainer(tuned_best_model)
        shap_values = explainer.shap_values(X_shap)

        if isinstance(shap_values, list):
            mean_abs_shap = np.mean([np.abs(values).mean(axis=0) for values in shap_values], axis=0)
        else:
            shap_array = np.array(shap_values)
            if shap_array.ndim == 3:
                mean_abs_shap = np.abs(shap_array).mean(axis=(0, 2))
            else:
                mean_abs_shap = np.abs(shap_array).mean(axis=0)

        shap_top_features = (
            pd.DataFrame({"feature": X_model.columns, "mean_abs_shap": mean_abs_shap})
            .sort_values("mean_abs_shap", ascending=False)
            .reset_index(drop=True)
        )
        shap_status = "Completed"
    else:
        shap_status = "Skipped - selected model is not a direct tree estimator"
except Exception as exc:
    shap_status = f"Skipped - {type(exc).__name__}: {exc}"

print(shap_status)
shap_top_features.head(20)


In [ ]:
plt.close("all")
# Plot SHAP summary importance if available.
if not shap_top_features.empty:
    plot_df = shap_top_features.head(20).sort_values("mean_abs_shap", ascending=True)
    ax = sns.barplot(data=plot_df, x="mean_abs_shap", y="feature", palette="rocket")
    ax.set_title(f"Top 20 SHAP Feature Importances - Tuned {best_model_name}")
    ax.set_xlabel("Mean Absolute SHAP Value")
    ax.set_ylabel("Feature")
    plt.tight_layout()
    plt.show()
else:
    print("SHAP feature plot skipped because SHAP output is unavailable.")


## 18.13 Drivers of Low, Neutral, and High CSAT

To identify class-specific drivers, a balanced one-vs-rest logistic model is trained on the same training data. Positive coefficients indicate features that increase the probability of a specific CSAT class relative to the others. This complements global importance charts by showing directional drivers for `Low`, `Neutral`, and `High` outcomes.


In [ ]:
# Class-specific directional drivers using one-vs-rest logistic regression.
ovr_model = OneVsRestClassifier(
    make_pipeline(
        StandardScaler(),
        LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
    )
)
ovr_model.fit(X_train, y_train)

class_driver_tables = {}
for class_label, estimator in zip(ovr_model.classes_, ovr_model.estimators_):
    logistic_step = estimator.named_steps["logisticregression"]
    coefficients = logistic_step.coef_.ravel()
    drivers_df = (
        pd.DataFrame({
            "feature": X_model.columns,
            "coefficient": coefficients,
            "abs_coefficient": np.abs(coefficients)
        })
        .sort_values("coefficient", ascending=False)
        .reset_index(drop=True)
    )
    class_driver_tables[class_label] = drivers_df

for class_label in class_labels:
    print("=" * 90)
    print(f"Top positive drivers for {class_label} CSAT")
    print("=" * 90)
    display(class_driver_tables[class_label].head(10)[["feature", "coefficient"]].round(4))


## 18.14 Business Recommendations for Flipkart Leadership

Based on the class imbalance, model evaluation, feature importance, and class-specific driver analysis, the following actions can help reduce low CSAT and improve customer satisfaction:

- Treat `Low` CSAT prediction as an early-warning workflow. Interactions predicted as high risk should receive faster escalation or supervisor review.
- Strengthen playbooks for issue categories and sub-categories that appear as positive drivers of low satisfaction.
- Review channel-level service quality, especially if email or other slower channels appear among low-CSAT drivers.
- Improve response-time discipline by monitoring delayed and severely delayed response flags.
- Use agent workload and shift-level signals to identify coaching and staffing needs.
- Improve onboarding and live support for newer or training agents if tenure-related features are associated with low CSAT.
- Improve data capture for sparse operational fields such as handling time, product category, item price, and city so future models are less dependent on imputation.
- Deploy the model as a decision-support layer, not an automated judgment system: predictions should trigger review, not punitive action.


## 18.15 Executive Summary for the 3-Class Benchmark

This executive summary describes the 3-class benchmark only. It is intentionally followed by Section 19, which evaluates whether the 3-class target should be used in production or replaced by a binary target. The key lesson from this benchmark is that weighted metrics can look acceptable while minority-class performance remains weak; therefore, Macro F1 and per-class recall are the correct decision metrics for this problem.


In [ ]:
# Executive summary table for the 3-class benchmark.
executive_summary = pd.DataFrame({
    "Section": [
        "Objective",
        "Dataset size",
        "Class distribution",
        "Best baseline model",
        "Tuned model",
        "Best CV Macro F1",
        "Final Test Macro F1",
        "Final Test Weighted F1",
        "Metric interpretation",
        "Top global driver source",
        "Business implication"
    ],
    "Summary": [
        "Benchmark a 3-class CSAT target: Low, Neutral, and High.",
        f"{X_model.shape[0]:,} rows and {X_model.shape[1]:,} leakage-aware features before modeling.",
        "; ".join([f"{row.csat_target}: {row['count']:,} ({row.percentage:.2f}%)" for _, row in class_distribution.iterrows()]),
        best_model_name,
        f"Tuned {best_model_name}",
        f"{search.best_score_:.4f}",
        f"{final_test_metrics['Macro F1']:.4f}",
        f"{final_test_metrics['Weighted F1']:.4f}",
        "Macro F1 is the primary metric because the Neutral class is very small and accuracy/weighted F1 can hide poor minority-class performance.",
        feature_importance_df.iloc[0]["feature"] if not feature_importance_df.empty else permutation_importance_df.iloc[0]["feature"],
        "The 3-class model is useful diagnostically, but the final deployment target should be chosen after the formulation comparison in Section 19."
    ]
})

executive_summary


### Final Model Workflow Outputs

At the end of this section, the notebook contains:

- `X_train`, `X_test`, `y_train`, `y_test`
- `comparison_table` ranked by Macro F1 and Weighted F1
- `classification_reports` and `confusion_matrices` for every model
- `best_model_name` and `best_model`
- `tuned_best_model` and `final_test_performance`
- `feature_importance_df`
- `permutation_importance_df`
- `shap_top_features` when SHAP is available
- `class_driver_tables` for Low, Neutral, and High CSAT drivers
- `executive_summary` for leadership review


# 19. Target Formulation Evaluation

The 3-class target (`Low`, `Neutral`, `High`) is business-readable, but it creates an extreme imbalance problem because `Neutral` is a very small class. This section compares the current 3-class formulation against two binary alternatives:

1. **High vs Non-High**: separates clearly satisfied customers from all others.
2. **Low vs Not-Low**: focuses directly on dissatisfied customers who need intervention.

The goal is to recommend the target formulation that best balances business usefulness and predictive performance.


## 19.1 Why Revisit the 3-Class Problem?

The 3-class target has a serious modeling challenge: the `Neutral` class is very small compared with `High`. In the previous model evaluation, the tuned Random Forest achieved reasonable weighted performance but weak Macro F1 because the model struggled to correctly identify the minority `Neutral` class.

For production use, a target should be both actionable and learnable. If a class is too small and ambiguous, it can reduce model reliability and make operational decisions harder.


In [ ]:
# Compare target distributions across alternative formulations.
formulation_df = pd.DataFrame({
    "csat_score": raw_df["CSAT Score"]
})

formulation_df["target_3_class"] = pd.cut(
    formulation_df["csat_score"],
    bins=[0, 2, 3, 5],
    labels=["Low", "Neutral", "High"],
    include_lowest=True
).astype(str)

formulation_df["target_high_vs_non_high"] = np.where(
    formulation_df["csat_score"] >= 4,
    "High",
    "Non-High"
)

formulation_df["target_low_vs_not_low"] = np.where(
    formulation_df["csat_score"] <= 2,
    "Low",
    "Not-Low"
)

formulation_distribution = []
for target_col in ["target_3_class", "target_high_vs_non_high", "target_low_vs_not_low"]:
    counts = formulation_df[target_col].value_counts()
    percentages = formulation_df[target_col].value_counts(normalize=True).mul(100).round(2)
    for label in counts.index:
        formulation_distribution.append({
            "formulation": target_col,
            "class": label,
            "count": counts[label],
            "percentage": percentages[label]
        })

formulation_distribution_df = pd.DataFrame(formulation_distribution)
formulation_distribution_df


## 19.2 Evaluation Design

To compare formulations fairly, the same leakage-aware feature matrix `X_model` is reused. Each target is split with stratification, then evaluated using the same two practical baseline models:

- Logistic Regression with class balancing
- Random Forest with class balancing

The comparison focuses on:

- **Macro F1** for class-balanced performance.
- **Balanced Accuracy** for average recall across classes.
- **Minority-class Recall** because the business cares about detecting risk segments, especially dissatisfied customers.
- **Weighted F1** for overall production performance.


In [ ]:
# Helper functions for binary and multi-class formulation comparison.
def build_formulation_target(csat_scores, formulation):
    if formulation == "3-Class Low/Neutral/High":
        return pd.cut(
            csat_scores,
            bins=[0, 2, 3, 5],
            labels=["Low", "Neutral", "High"],
            include_lowest=True
        ).astype(str)
    if formulation == "High vs Non-High":
        return pd.Series(np.where(csat_scores >= 4, "High", "Non-High"), index=csat_scores.index)
    if formulation == "Low vs Not-Low":
        return pd.Series(np.where(csat_scores <= 2, "Low", "Not-Low"), index=csat_scores.index)
    raise ValueError(f"Unknown formulation: {formulation}")


def minority_recall_score(y_true, y_pred):
    counts = pd.Series(y_true).value_counts()
    minority_label = counts.idxmin()
    labels = sorted(pd.Series(y_true).unique())
    recall_values = recall_score(y_true, y_pred, labels=labels, average=None, zero_division=0)
    recall_lookup = dict(zip(labels, recall_values))
    return minority_label, recall_lookup[minority_label]


def evaluate_target_formulation(formulation_name, target_series):
    X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
        X_model,
        target_series.astype(str),
        test_size=0.20,
        random_state=RANDOM_STATE,
        stratify=target_series.astype(str)
    )

    candidate_models = {
        "Logistic Regression": make_pipeline(
            StandardScaler(),
            LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=80,
            class_weight="balanced_subsample",
            max_depth=18,
            min_samples_leaf=10,
            n_jobs=1,
            random_state=RANDOM_STATE
        )
    }

    rows = []
    reports = {}
    matrices = {}
    for model_name, estimator in candidate_models.items():
        estimator.fit(X_train_f, y_train_f)
        y_pred_f = estimator.predict(X_test_f)
        minority_label, minority_recall = minority_recall_score(y_test_f, y_pred_f)

        row = evaluate_predictions(f"{formulation_name} - {model_name}", y_test_f, y_pred_f)
        row.update({
            "Formulation": formulation_name,
            "Base Model": model_name,
            "Class Count": target_series.nunique(),
            "Minority Class": minority_label,
            "Minority Recall": minority_recall,
            "Majority Class Share": target_series.value_counts(normalize=True).max()
        })
        rows.append(row)
        reports[f"{formulation_name} - {model_name}"] = classification_report(y_test_f, y_pred_f, zero_division=0, output_dict=True)
        matrices[f"{formulation_name} - {model_name}"] = confusion_matrix(y_test_f, y_pred_f, labels=sorted(target_series.unique()))

    return rows, reports, matrices

formulation_names = [
    "3-Class Low/Neutral/High",
    "High vs Non-High",
    "Low vs Not-Low"
]

formulation_results = []
formulation_reports = {}
formulation_matrices = {}

for formulation_name in formulation_names:
    target_series = build_formulation_target(raw_df["CSAT Score"], formulation_name)
    rows, reports, matrices = evaluate_target_formulation(formulation_name, target_series)
    formulation_results.extend(rows)
    formulation_reports.update(reports)
    formulation_matrices.update(matrices)

formulation_comparison_table = (
    pd.DataFrame(formulation_results)
    .sort_values(["Macro F1", "Weighted F1"], ascending=False)
    .reset_index(drop=True)
)

selected_cols = [
    "Formulation", "Base Model", "Class Count", "Accuracy", "Balanced Accuracy",
    "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1",
    "Minority Class", "Minority Recall", "Majority Class Share"
]
formulation_comparison_table[selected_cols]


## 19.3 Best Formulation by Predictive Performance


In [ ]:
# Best model per target formulation.
best_by_formulation = (
    formulation_comparison_table
    .sort_values(["Formulation", "Macro F1", "Weighted F1"], ascending=[True, False, False])
    .groupby("Formulation", as_index=False)
    .head(1)
    .sort_values(["Macro F1", "Weighted F1"], ascending=False)
    .reset_index(drop=True)
)

best_by_formulation[selected_cols]


In [ ]:
plt.close("all")
# Compare best Macro F1 and Weighted F1 by formulation.
plot_df = best_by_formulation.melt(
    id_vars=["Formulation"],
    value_vars=["Macro F1", "Weighted F1", "Balanced Accuracy"],
    var_name="Metric",
    value_name="Score"
)

ax = sns.barplot(data=plot_df, x="Formulation", y="Score", hue="Metric", palette="Set2")
ax.set_title("Best Model Performance by Target Formulation")
ax.set_xlabel("Target Formulation")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Metric")

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", padding=3)

plt.tight_layout()
plt.show()


## 19.4 Recommendation

The recommended formulation is **Low vs Not-Low**.

### Why not keep the 3-class problem?

The 3-class setup is intuitive, but it is not the best production target for this dataset. The `Neutral` class is extremely small and behaviorally ambiguous. In model results, the 3-class workflow struggles to identify `Neutral`, which depresses Macro F1 and creates unstable class-specific predictions. A model that cannot reliably distinguish the smallest class should not drive operational decisions for that class.

### Why not High vs Non-High?

`High vs Non-High` produces the highest Macro F1 in the comparison by a narrow margin, but it blends truly dissatisfied customers with neutral customers. That weakens business actionability: a neutral score does not require the same urgency, escalation, or service recovery effort as a score 1 or 2 failure.

### Why Low vs Not-Low is better

`Low vs Not-Low` directly answers the most valuable business question: **which interactions are at risk of producing dissatisfied customers?** It supports escalation, coaching, root-cause analysis, and customer recovery workflows. It is also more learnable than the 3-class problem because it removes the tiny neutral class while preserving the most important business-risk segment.

The slight Macro F1 tradeoff versus `High vs Non-High` is acceptable because `Low vs Not-Low` is more aligned with the intervention objective: reduce low CSAT cases, not merely separate high scores from everything else.


In [ ]:
# Final recommendation summary.
recommended_formulation = "Low vs Not-Low"
recommended_row = (
    best_by_formulation[best_by_formulation["Formulation"] == recommended_formulation]
    .iloc[0]
)

formulation_recommendation_summary = pd.DataFrame({
    "Decision Area": [
        "Recommended formulation",
        "Business objective",
        "Why 3-class is not preferred",
        "Why High vs Non-High is not preferred",
        "Recommended model from comparison",
        "Macro F1",
        "Weighted F1",
        "Balanced Accuracy",
        "Minority class",
        "Minority recall"
    ],
    "Recommendation": [
        recommended_formulation,
        "Detect and reduce dissatisfied customer experiences before they become repeat service failures.",
        "The Neutral class is too small and ambiguous, leading to weak minority-class learning.",
        "It blends neutral and dissatisfied customers, reducing actionability for service recovery.",
        recommended_row["Base Model"],
        f"{recommended_row['Macro F1']:.4f}",
        f"{recommended_row['Weighted F1']:.4f}",
        f"{recommended_row['Balanced Accuracy']:.4f}",
        recommended_row["Minority Class"],
        f"{recommended_row['Minority Recall']:.4f}"
    ]
})

formulation_recommendation_summary


## 19.5 Final Decision for the ML Notebook

For the next production ML notebook, use **binary classification: `Low vs Not-Low`**.

This formulation best balances business usefulness and predictive performance. It focuses the model on preventing low customer satisfaction, avoids the unstable tiny `Neutral` class, and produces a clearer operational output: interactions predicted as `Low` should be prioritized for intervention, escalation, or service-quality review.

The 3-class model should remain in the notebook as a benchmark and diagnostic exercise, but it should not be the first deployment candidate unless more neutral-class data becomes available or the business explicitly needs a separate neutral workflow.


# 20. Lead Data Scientist Review

This final review evaluates the notebook from the perspective of a lead data scientist assessing intern work for stipend allocation. The notebook is strong overall: it includes a complete EDA, documented data quality checks, a reproducible ML-preparation pipeline, leakage-aware modeling, class imbalance analysis, model comparison, tuning, explainability, target-formulation evaluation, and business recommendations.

The main improvement made in this review is to clarify the difference between the 3-class benchmark and the final production recommendation. The notebook now makes it explicit that the 3-class model is useful for analysis, while `Low vs Not-Low` is the preferred deployment target.


## 20.1 Remaining Weaknesses, Shortcuts, and Limitations

- **Saved outputs:** The notebook is executable, but rendered outputs may not be persisted unless it is run in Jupyter or an equivalent notebook environment.
- **Optional libraries:** XGBoost, LightGBM, and SHAP are handled gracefully when unavailable, but installing them could improve model comparison and explainability depth.
- **Target-derived aggregate risk:** `agent_historical_avg_csat` was correctly removed from modeling because it is target-derived. Any future historical performance features must be computed using only past data or within training folds.
- **High missingness:** `connected_handling_time`, `Customer_City`, `Product_category`, `Item_price`, and `order_date_time` have substantial missingness. The notebook handles this systematically, but the business should improve source-system data capture.
- **Neutral class weakness:** The 3-class setup is not production-preferred because the neutral class is too small and model recall is weak.
- **Temporal validation:** The current split is stratified random. For production, a time-based validation split should be added once more months of data are available.
- **Causal interpretation:** Feature importance and coefficients identify associations, not causal drivers. They should guide investigation, not be treated as proof of cause.
- **Operational cost tradeoff:** The notebook reports standard ML metrics, but a production decision should also define the cost of false positives versus false negatives for low-CSAT intervention.


## 20.2 Metric Interpretation Check

The notebook correctly avoids relying only on accuracy because the target is imbalanced. For the 3-class benchmark, weighted F1 is higher than Macro F1 because the model performs much better on the dominant `High` class than on the minority `Neutral` class. This is why Macro F1, balanced accuracy, and minority-class recall are emphasized.

For the target-formulation comparison, `High vs Non-High` has a slightly higher Macro F1, but `Low vs Not-Low` is recommended because it is more operationally useful. This is an acceptable data-science decision: the best production target is not always the one with the single highest metric if another formulation better supports business action.


## 20.3 Visualization and Insight Review

All 16 EDA visualizations have accompanying business insight markdown. The visual narrative is coherent: it moves from overall CSAT distribution to channel, issue category, product category, handling time, agent, city, time trend, and numeric correlation analysis.

The insights are appropriately cautious where data is sparse, especially for `connected_handling_time`, `Customer_City`, and `Product_category`. This is important because overclaiming from sparse fields is a common intern-level mistake.


## 20.4 Refined Deployment Recommendation

Deploy the model as a **decision-support system** for identifying likely `Low` CSAT interactions, not as an automated punitive or ranking system for agents.

Recommended production target:

- `Low`: CSAT 1-2
- `Not-Low`: CSAT 3-5

Recommended operating workflow:

- Score support interactions after enough non-leaking features are available.
- Route high-risk `Low` predictions to a supervisor queue, service recovery queue, or quality audit sample.
- Use predictions to prioritize coaching, not to penalize agents automatically.
- Track model decisions and eventual CSAT outcomes to build a feedback loop.
- Retrain and recalibrate the model regularly as support policies, customer behavior, and product mix change.

Before deployment, leadership should define the acceptable tradeoff between false positives and false negatives. Missing a truly low-CSAT interaction may be costlier than reviewing a few extra cases, so recall for the `Low` class should be a key operating metric.


# 21. Production Deployment Considerations

## Monitoring

Monitor both model performance and operational impact after deployment. Key metrics should include `Low`-class recall, precision, Macro F1, false negative rate for low-CSAT cases, predicted low-CSAT rate by channel, and intervention conversion rate. Model monitoring should be paired with business KPIs such as repeat contacts, escalation rate, refund/return friction, and post-intervention CSAT recovery.

## Model Retraining Frequency

Start with monthly retraining if fresh CSAT labels are available. Move to bi-weekly retraining during major operational changes such as sale events, policy changes, new support workflows, or changes in delivery/returns partners. If data volume is low for some segments, use a rolling 2-3 month training window to stabilize minority-class learning.

## Data Drift Detection

Track feature drift and prediction drift. Important drift checks include channel mix, issue category mix, sub-category mix, response-time distribution, agent tenure distribution, city coverage, missing-value rates, and predicted `Low` probability distribution. Population Stability Index, Jensen-Shannon divergence, or simpler threshold-based monitoring can be used depending on platform maturity.

## Fairness Considerations

The model should not be used to unfairly penalize agents, cities, shifts, or tenure groups. Some segments may receive harder cases, have different operational constraints, or suffer from missing data. Fairness monitoring should compare error rates and low-CSAT prediction rates across channels, shifts, tenure buckets, supervisors, and major city groups. If agent-level outputs are used, they must be adjusted for case mix and reviewed by humans.

## Limitations of the Current Approach

The current model is based on one month of data and uses a random stratified split rather than a true forward-looking temporal validation. Several potentially useful fields are highly missing. Free-text customer remarks are not modeled with NLP. Optional advanced libraries were not available in the local runtime. Finally, model explanations show associations, not causal proof.

## Production Readiness Verdict

The notebook is submission-ready and suitable as a strong intern-level ML project. For real deployment, the recommended next step is a dedicated production notebook or pipeline using the `Low vs Not-Low` target, time-based validation, calibrated probabilities, threshold optimization, richer monitoring, and a human-in-the-loop intervention workflow.


In [ ]:
# Final production recommendation object for downstream handoff.
production_deployment_recommendation = {
    "recommended_target": "Low vs Not-Low",
    "positive_class": "Low",
    "business_goal": "Identify interactions at risk of low CSAT so Flipkart can intervene earlier.",
    "primary_model_selection_metric": "Macro F1",
    "operating_metric_to_monitor": "Low-class recall and false negative rate",
    "deployment_mode": "Human-in-the-loop decision support",
    "not_recommended_use": "Automated punitive agent ranking or fully automated customer decisions",
    "minimum_next_steps": [
        "Use time-based validation on additional months of data",
        "Calibrate predicted probabilities",
        "Choose an intervention threshold based on cost of false negatives vs false positives",
        "Implement drift and fairness monitoring",
        "Retrain monthly or when drift is detected"
    ]
}

production_deployment_recommendation
